# RAG Hybrid Retrieval + Generation
This workbook allows to perform the first 3 RAGs as discussed.\
RAG1= dense or BM25 RAG2 = hybrid (dense + BM25). RAG3 = hybrid + reranking. .\
Set USE_RERANKING in Cell 1 to switch. 
Output: input, generated_output, retrieved_context.

In [6]:
from pathlib import Path

CWD = Path.cwd().resolve()

if CWD.as_posix().startswith("/workspace"):
    DATA_ROOT = Path("/workspace/data")
else:
    DATA_ROOT = Path(r"C:\Users\chenq\smu-llm-group-project\data")

if (CWD / "test_final.csv").exists():
    CSV_PATH = CWD / "test_final.csv"
else:
    CSV_PATH = DATA_ROOT / "test_final.csv"

CHUNK_STRATEGY = "section_400token" # replace with fixed_500char if needed 
N_QUESTIONS = 50    #max size of the test set is 300
OUTPUT_CSV = Path("rag_eval_output.csv")

# False = RAG2 (hybrid only), True = RAG3 (hybrid + reranking)
USE_RERANKING = False

print("CWD:", CWD)
print("DATA_ROOT:", DATA_ROOT)
print("CSV_PATH:", CSV_PATH, "exists?", CSV_PATH.exists())
print("USE_RERANKING (RAG3):", USE_RERANKING)

CWD: /workspace/notebooks
DATA_ROOT: /workspace/data
CSV_PATH: /workspace/notebooks/test_final.csv exists? True
USE_RERANKING (RAG3): False


In [2]:
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
candidates = [CWD / "src", CWD.parent / "src"]
for _ in range(5):
    if (CWD / "src" / "data_loader.py").exists():
        candidates.append(CWD / "src")
        break
    CWD = CWD.parent

src_path = None
for p in candidates:
    p = p.resolve()
    if (p / "data_loader.py").exists():
        src_path = p
        break
if src_path is None:
    raise FileNotFoundError("Could not find src/ with data_loader.py")
sys.path.insert(0, str(src_path))
print("Using src:", src_path)

from data_loader import load_and_sample_test_set
from answer import format_context, generate_answer
from bm25 import BM25Retriever, load_bm25_index
from embedding_manager import MedicalEmbeddingManager
from hybrid import hybrid_retrieve
import pandas as pd

Using src: /workspace/src


/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# 1) Load and sample test set
test_df = load_and_sample_test_set(CSV_PATH, n_total=N_QUESTIONS, random_state=42)
print("Sampled questions:", len(test_df))

# 2) Load chunks: from file or build from Chroma
chunks_dir = DATA_ROOT / "chunks"
parquet_path = chunks_dir / f"{CHUNK_STRATEGY}_chunks.parquet"
csv_path = chunks_dir / f"{CHUNK_STRATEGY}_chunks.csv"

if parquet_path.exists():
    chunks_df = pd.read_parquet(parquet_path)
    print("Loaded chunks from:", parquet_path)
elif csv_path.exists():
    chunks_df = pd.read_csv(csv_path)
    print("Loaded chunks from:", csv_path)
else:
    import chromadb

    chroma_path = DATA_ROOT / "vector_store" / CHUNK_STRATEGY / "pubmedbert"
    if not chroma_path.exists():
        raise FileNotFoundError(f"Chroma path not found: {chroma_path}")

    client = chromadb.PersistentClient(path=str(chroma_path))
    coll = client.get_or_create_collection("medical_rag", metadata={"hnsw:space": "cosine"})
    all_docs = coll.get(include=["documents", "metadatas"])

    ids = all_docs["ids"]
    docs = all_docs["documents"]
    metas = all_docs.get("metadatas") or [{}] * len(ids)

    def idx(i):
        m = metas[i] if i < len(metas) else {}
        mid = m.get("chunk_idx", ids[i])
        if isinstance(mid, (int, float)):
            return int(mid)
        if isinstance(mid, str) and mid.isdigit():
            return int(mid)
        return i

    order = sorted(range(len(ids)), key=idx)
    chunk_texts = [docs[i] if i < len(docs) else "" for i in order]
    chunks_df = pd.DataFrame({"chunk_text": chunk_texts})
    for key in (metas[0].keys() if metas else []):
        chunks_df[key] = [metas[i].get(key) if i < len(metas) else None for i in order]
    print("Built chunks_df from Chroma, rows:", len(chunks_df))

chunk_text_col = "chunk_text" if "chunk_text" in chunks_df.columns else chunks_df.columns[0]

# 3) Dense retriever (768-dim via MedicalEmbeddingManager)
import chromadb

_chroma_path = DATA_ROOT / "vector_store" / CHUNK_STRATEGY / "pubmedbert"
if not _chroma_path.exists():
    raise FileNotFoundError(f"Chroma path not found: {_chroma_path}")

dense_client = chromadb.PersistentClient(path=str(_chroma_path))
dense_coll = dense_client.get_or_create_collection("medical_rag", metadata={"hnsw:space": "cosine"})
embedder = MedicalEmbeddingManager(model_name="pubmedbert")

def dense_fn(q: str, n: int):
    vec = embedder.embed_query(q)
    if hasattr(vec, "tolist"):
        vec = vec.tolist()
    query_emb = [vec]
    res = dense_coll.query(
        query_embeddings=query_emb,
        n_results=n,
        include=["documents", "metadatas"],
    )
    docs = res["documents"][0]
    metas = res["metadatas"][0] if res.get("metadatas") else [{}] * len(docs)
    out = []
    for i, (doc, meta) in enumerate(zip(docs, metas)):
        chunk_id = meta.get("chunk_idx", i)
        out.append((chunk_id, doc, meta))
    return out

# 4) BM25 retriever
bm25_index = load_bm25_index(DATA_ROOT / "bm25_index" / f"{CHUNK_STRATEGY}.pkl")
bm25 = BM25Retriever(bm25_index, chunks_df, chunk_text_column=chunk_text_col)

def bm25_fn(q: str, n: int):
    return bm25.query(q, top_k=n)

Sampled questions: 50


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built chunks_df from Chroma, rows: 109234


/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Cell 4 (Code) – Reranker (used only when USE_RERANKING is True)

In [4]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)

def rerank_chunks(question: str, chunks: list, top_k: int = 10):
    if not chunks:
        return []
    texts = [c[1] for c in chunks]
    pairs = [(question, t) for t in texts]
    scores = reranker.predict(pairs)
    scored = list(zip(chunks, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [c for c, _ in scored[:top_k]]

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
rows = []
n_final_retrieve = 20 if USE_RERANKING else 10

for i, row in test_df.iterrows():
    q = row["question"]
    if (i + 1) % 20 == 0 or i == 0:
        label = "RAG3" if USE_RERANKING else "RAG2"
        print(f"Processing {i+1}/{len(test_df)} ({label})")

    chunks = hybrid_retrieve(q, dense_fn, bm25_fn, n_dense=40, n_bm25=40, n_final=n_final_retrieve)
    if USE_RERANKING:
        chunks = rerank_chunks(q, chunks, top_k=10)

    context = format_context(chunks, sep="\n\n---\n\n")
    answer = generate_answer(q, context, model_name="google/flan-t5-small")
    rows.append({"input": q, "generated_output": answer, "retrieved_context": context})

result_df = pd.DataFrame(rows)
out_path = Path("rag3_eval_output.csv") if USE_RERANKING else OUTPUT_CSV
result_df.to_csv(out_path, index=False, encoding="utf-8")
print("Saved to:", out_path.resolve())
result_df.head()

Processing 1/50 (RAG2)


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
/opt/conda/lib/python3.10/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Processing 20/50 (RAG2)
Processing 40/50 (RAG2)
Saved to: /workspace/notebooks/rag_eval_output.csv


,input,generated_output,retrieved_context
0,What are the genetic variants in Turner syndrome?,idiopathic,"Respiratory syncytial virus: how, why and what..."
1,What proportion of colorectal cancer cases are...,50%,To identify the risk factors responsible for c...
2,Most common reversible cause of infertility in...,dysphagia,"Respiratory syncytial virus: how, why and what..."
3,What is the function of CD47,ryanodine receptor type 2 (RyR2) dysfunction,An overview of normal pressure hydrocephalus a...
4,Which computational method exists for predicti...,RTN2B,"After completion of this article, the reader s..."
